In [1]:
from pathlib import Path
import pandas as pd
import tempfile
import boto3

In [2]:
access_key_id = "CXZYSZTWNIKRILBHZXWH"
access_secret = "WIXGDORKWINEIDIIYBDRYRCODZIQFBDWYYXDQIGC"
S3_ENDPOINT_URL = "https://bright-cape-b-v--bright-cape-x-belimo-c4c.eu-1.celonis.cloud/api/data-ingestion"
S3_BUCKET = "continuous"
S3_CONNECTION_ID = "52a27f37-bb72-40ec-bed4-c09a6f703515"

# Loading Data to Celonis

In [ ]:
# The status file written by AgenticSignatureVerification.ipynb
DOCUMENT_STATUS_PATH = Path.cwd() / "DocumentStatus.xlsx"
DOCUMENT_STATUS_TABLE = "DocumentStatus"

In [ ]:
def upload_local_file_to_s3(
    file_path: Path,
    object_name: str,
    s3_client,
 ):
    """Upload a local file with one request to the Celonis S3 endpoint."""

    with file_path.open("rb") as file:
        s3_client.put_object(
            Bucket=S3_BUCKET,
            Key=object_name,
            Body=file,
            ContentLength=file_path.stat().st_size,
        )

    print(f"Uploaded {file_path.name} to {S3_BUCKET}/{object_name}")



def upload_dataframe_to_s3(
    dataframe_name: str,
    dataframe,
    s3_client,
 ):
    """Write one dataframe locally as Parquet and upload it to Celonis."""

    object_name = (
        f"connection/{S3_CONNECTION_ID}/{dataframe_name}/"
        f"{dataframe_name}.parquet"
    )

    upload_dataframe = dataframe.copy()
    object_columns = upload_dataframe.columns[
        upload_dataframe.dtypes == "object"
    ]
    for column in object_columns:
        upload_dataframe[column] = upload_dataframe[column].astype("string")

    with tempfile.TemporaryDirectory() as temporary_directory:
        local_file = Path(temporary_directory) / f"{dataframe_name}.parquet"
        upload_dataframe.to_parquet(local_file, index=False)
        upload_local_file_to_s3(local_file, object_name, s3_client)



def upload_document_status_to_s3(
    document_status_path: Path,
    celonis_access_key: str,
    celonis_access_secret: str,
 ):
    """Read DocumentStatus.xlsx and upload it to Celonis as the DocumentStatus table."""

    if not document_status_path.is_file():
        raise FileNotFoundError(
            f"Document status file does not exist: {document_status_path}"
        )

    s3_client = boto3.client(
        "s3",
        endpoint_url=S3_ENDPOINT_URL,
        aws_access_key_id=celonis_access_key,
        aws_secret_access_key=celonis_access_secret,
    )

    # Keep every column as a string, the same way the verification notebook reads it
    dataframe = pd.read_excel(document_status_path, dtype=str)
    upload_dataframe_to_s3(DOCUMENT_STATUS_TABLE, dataframe, s3_client)
    print(f"Finished uploading {len(dataframe)} row(s) of {DOCUMENT_STATUS_TABLE}")


In [ ]:
upload_document_status_to_s3(
    document_status_path=DOCUMENT_STATUS_PATH,
    celonis_access_key=access_key_id,
    celonis_access_secret=access_secret,
)